In [ ]:
# import libraries

import numpy as np
import seaborn as sns
sns.set(color_codes=True)
import matplotlib.pyplot as plt
%matplotlib inline
import pickle
import matplotlib.pyplot as plt
import numpy as np

data_path = "/data/test_newrepo/"
file_name = "run52_hard_mega_shared"
normalize= 0
numpy_file=0
number_of_detectors = 6

In [ ]:
file_path = data_path+'/'+file_name+'_dataset.pkl'

# Load the array from the pickle file
with open(file_path, 'rb') as file:
    data_array = pickle.load(file)

In [ ]:
def process_dataset(data_array):

    dataset = np.empty((len(data_array),number_of_detectors))
    labels = np.empty((len(data_array),2))
    
    count = -1
    for element in data_array:
        
        count+=1
        
        dataset[count] = element['counts']
        labels[count][0] = float(element['coord'][0])
        labels[count][1] = float(element['coord'][1])
        
    labels = labels[:count+1]
    dataset = dataset[:count+1]

    #scale labels
    max_lon = 180.0
    min_lon = 0.0
    
    max_lat = 360.0
    min_lat = 0.0
    
    labels_norm = np.empty((len(dataset),2))
    
    count = -1
    for element in labels:
        
        count+=1
        if normalize==1:
            labels_norm[count][0] = (labels[count][0]-min_lon)/(max_lon-min_lon)
            labels_norm[count][1] = (labels[count][1]-min_lat)/(max_lat-min_lat)
        else:
            labels_norm[count][0] = labels[count][0]
            labels_norm[count][1] = labels[count][1]

    # Convert to radians
    coords_rad = []
    
    for theta, phi in labels:
        #if lon > 90 and lon < 270:
            
        coords_rad.append([theta, phi])
        #else:
        #    continue

    theta, phi = zip(*coords_rad)
    
   
    return dataset, labels, labels_norm, theta, phi, coords_rad


In [ ]:
dataset, labels, labels, theta, phi, coords_rad = process_dataset(data_array)

In [ ]:
counts = np.empty((len(dataset),number_of_detectors))
index = -1
for element in dataset:
    index=index+1
    counts[index] = element

In [ ]:
# Define 5-degree bins
bins = np.arange(0, 185, 5)

# Group counts by 5-degree bins
grouped_counts = np.zeros((len(bins), number_of_detectors))
counts_per_bin = np.zeros((len(bins), number_of_detectors))

total_count = 0
for i in range(0, number_of_detectors):
    for theta, phi, count in zip(labels[:, 0], labels[:, 1], counts[:, i]):
        
        if (phi > -5 and phi < 5):
            total_count += 1
            bin_index = int(theta // 5)
            grouped_counts[bin_index][i] = grouped_counts[bin_index][i] + count
            counts_per_bin[bin_index][i] += 1

# Compute the mean counts in each bin
grouped_counts = grouped_counts[:total_count]
counts_per_bin = counts_per_bin[:total_count]

# Plot histogram
plt.figure(figsize=(10, 6))
#plt.bar(bins, grouped_counts[:,0]/counts_per_bin[:,0], width=5, align='edge', alpha=.5, label="z1")
#plt.bar(bins, grouped_counts[:,1]/counts_per_bin[:,1], width=5, align='edge', alpha=.5, label="z0")
plt.bar(bins, grouped_counts[:,2]/counts_per_bin[:,2], width=5, align='edge', alpha=.5, label="BGO_X1")
plt.bar(bins, grouped_counts[:,3]/counts_per_bin[:,3], width=5, align='edge', alpha=.5, label="BGO_X0")
#plt.bar(bins, grouped_counts[:,4]/counts_per_bin[:,4], width=5, align='edge', alpha=.5, label="BGO_Y1")
#plt.bar(bins, grouped_counts[:,5]/counts_per_bin[:,5], width=5, align='edge', alpha=.5, label="BGO_Y0")
plt.title('')
plt.grid(True)
plt.xlabel('Theta (°)', fontsize=15)
plt.ylabel('Panel mean counts', fontsize=15)
plt.tick_params(axis='both', labelsize=15) 
plt.legend(fontsize=15)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define 5-degree bins
bins = np.arange(0, 185, 5)
n_bins = len(bins)
n_panels = number_of_detectors

# Initialize lists for each bin and each panel
grouped_counts = [[[] for _ in range(n_panels)] for _ in range(n_bins)]

# Collect counts into bins
for i in range(n_panels):
    for theta, phi, count in zip(labels[:, 0], labels[:, 1], counts[:, i]):
        if 355 < phi or phi < 5:
            bin_index = int(theta // 5)
            if bin_index < n_bins:
                grouped_counts[bin_index][i].append(count)

# Compute mean and standard deviation for each bin and panel
mean_counts = np.zeros((n_bins, n_panels))
std_dev = np.zeros((n_bins, n_panels))

for b in range(n_bins):
    for p in range(n_panels):
        data = grouped_counts[b][p]
        if len(data) > 0:
            mean_counts[b][p] = np.mean(data)
            std_dev[b][p] = 3 * np.std(data)   # 3σ error bars
        else:
            mean_counts[b][p] = np.nan
            std_dev[b][p] = 0

# Plot
plt.figure(figsize=(10, 6))
width = 5  # slightly less than 5 to separate panels visually

# Offsets for different panels in the bar plot (not used here but can be added)
offsets = [-2, -1, 0, 1, 2, 3]

panel_labels = ["z1", "z0", "BGO_X1", "BGO_X0", "BGO_Y1", "BGO_Y0"]
colors = ['b', 'g', 'r', 'c', 'm', 'y']

for i in range(2, 6):  # Adjust range if you want to show all panels
    x_pos = bins  # + offsets[i] if you want panel separation
    plt.bar(
        x_pos, 
        mean_counts[:, i], 
        width=width, 
        alpha=0.5, 
        label=panel_labels[i], 
        yerr=std_dev[:, i], 
        capsize=3, 
        color=colors[i]
    )

plt.xlabel('Theta (°)')
plt.ylabel('Panel mean counts')
plt.title('')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Define 5-degree bins
bins = np.arange(0, 185, 5)

# Group counts by 5-degree bins
grouped_counts = np.zeros((len(bins), number_of_detectors))
counts_per_bin = np.zeros((len(bins), number_of_detectors))

total_count = 0
for i in range(0, number_of_detectors):
    for theta, phi, count in zip(labels[:, 0], labels[:, 1], counts[:, i]):
        
        if (phi > -5 and phi < 5):
            total_count += 1
            bin_index = int(theta // 5)
            grouped_counts[bin_index][i] = grouped_counts[bin_index][i] + count
            counts_per_bin[bin_index][i] += 1

# Compute the mean counts in each bin
grouped_counts = grouped_counts[:total_count]
counts_per_bin = counts_per_bin[:total_count]

# Plot histogram
plt.figure(figsize=(10, 6))
plt.bar(bins, grouped_counts[:, 0] / counts_per_bin[:, 0], width=5, align='edge', alpha=.5, label="BGO_Z1")
plt.bar(bins, grouped_counts[:, 1] / counts_per_bin[:, 1], width=5, align='edge', alpha=.5, label="BGO_Z0")
#plt.bar(bins, grouped_counts[:, 2] / counts_per_bin[:, 2], width=5, align='edge', alpha=.5, label="BGO_X1")
#plt.bar(bins, grouped_counts[:, 3] / counts_per_bin[:, 3], width=5, align='edge', alpha=.5, label="BGO_X0")
#plt.bar(bins, grouped_counts[:, 4] / counts_per_bin[:, 4], width=5, align='edge', alpha=.5, label="BGO_Y1")
#plt.bar(bins, grouped_counts[:, 5] / counts_per_bin[:, 5], width=5, align='edge', alpha=.5, label="BGO_Y0")

plt.xlabel('Theta (°)', fontsize=15)
plt.ylabel('Panel mean counts', fontsize=15)
plt.tick_params(axis='both', labelsize=15) 
plt.legend(fontsize=15, loc="upper left")
plt.title('')
plt.grid(True)

plt.show()


In [ ]:
for phi_angle in range(0, 360, 10):
    
    print(phi_angle)
    if phi_angle == 360:
        continue

    # Define 5-degree bins
    bins = np.arange(0, 185, 5)

    # Group counts by 5-degree bins
    grouped_counts = np.zeros((len(bins), number_of_detectors))
    counts_per_bin = np.zeros((len(bins), number_of_detectors))

    total_count = 0
    for i in range(0, number_of_detectors):
        for theta, phi, count in zip(labels[:, 0], labels[:, 1], counts[:, i]):
            
            if (phi > phi_angle and phi < phi_angle + 10):
                total_count += 1
                bin_index = int(theta // 5)
                grouped_counts[bin_index][i] = grouped_counts[bin_index][i] + count
                counts_per_bin[bin_index][i] += 1

    # Compute the mean counts in each bin
    grouped_counts = grouped_counts[:total_count]
    counts_per_bin = counts_per_bin[:total_count]

    # Plot histogram
    plt.figure(figsize=(10, 6))
    plt.bar(bins, grouped_counts[:, 0] / counts_per_bin[:, 0] / total_count, width=5, align='edge', alpha=.5, label="BGO_Z1")
    plt.bar(bins, grouped_counts[:, 1] / counts_per_bin[:, 1] / total_count, width=5, align='edge', alpha=.5, label="BGO_Z0")
    plt.bar(bins, grouped_counts[:, 2] / counts_per_bin[:, 2] / total_count, width=5, align='edge', alpha=.5, label="X1")
    plt.bar(bins, grouped_counts[:, 3] / counts_per_bin[:, 3] / total_count, width=5, align='edge', alpha=.5, label="X0")
    plt.bar(bins, grouped_counts[:, 4] / counts_per_bin[:, 4] / total_count, width=5, align='edge', alpha=.5, label="Y1")
    plt.bar(bins, grouped_counts[:, 5] / counts_per_bin[:, 5] / total_count, width=5, align='edge', alpha=.5, label="Y0")
    plt.xlabel('Theta (°)')
    plt.ylabel('Normalized panel mean counts')
    plt.title('')
    plt.grid(True)
    plt.legend(loc='upper left')
    plt.show()


In [ ]:
# Define 5-degree bins
bins = np.arange(0, 365, 5)

# Group counts by 5-degree bins
grouped_counts = np.zeros((len(bins), 6))
counts_per_bin = np.zeros((len(bins), 6))

total_count = 0
for i in range(0, 6):
    for theta, phi, count in zip(labels[:, 0], labels[:, 1], counts[:, i]):
        
        if (theta > 85 and theta < 95):
            total_count += 1
            bin_index = int(phi // 5)
            grouped_counts[bin_index][i] = grouped_counts[bin_index][i] + count
            counts_per_bin[bin_index][i] += 1

# Compute the mean counts in each bin
grouped_counts = grouped_counts[:total_count]
counts_per_bin = counts_per_bin[:total_count]

# Plot histogram
plt.figure(figsize=(10, 6))
#plt.bar(bins, grouped_counts[:, 0] / counts_per_bin[:, 0], width=5, align='edge', alpha=.5, label="Bottom")
#plt.bar(bins, grouped_counts[:, 1] / counts_per_bin[:, 1], width=5, align='edge', alpha=.5, label="X1")
plt.bar(bins, grouped_counts[:, 2] / counts_per_bin[:, 2], width=5, align='edge', alpha=.5, label="BGO_X1")
plt.bar(bins, grouped_counts[:, 3] / counts_per_bin[:, 3], width=5, align='edge', alpha=.5, label="BGO_X0")
plt.bar(bins, grouped_counts[:, 4] / counts_per_bin[:, 4], width=5, align='edge', alpha=.5, label="BGO_Y1")
plt.bar(bins, grouped_counts[:, 5] / counts_per_bin[:, 5], width=5, align='edge', alpha=.5, label="BGO_Y0")

plt.xlabel('Phi (°)', fontsize=15)
plt.ylabel('Panel mean counts', fontsize=15)
plt.tick_params(axis='both', labelsize=15) 
plt.legend(fontsize=15)
plt.title('')
plt.grid(True)

plt.show()


In [ ]:
# Definisci i bin di 5 gradi
bin_angolari = np.arange(0, 365, 5)

# Raggruppa i conteggi in base ai bin di 5 gradi
conteggi_raggruppati = np.zeros((len(bin_angolari), 6))
conteggi_per_bin = np.zeros((len(bin_angolari), 6))

conteggio_totale = 0
for i in range(0, 6):
    for angolo_theta, angolo_phi, conteggio in zip(labels[:, 0], labels[:, 1], counts[:, i]):
        # X tra 5 e 20, Y tra 20 e 30
        if angolo_theta < 35:
            conteggio_totale += 1
            indice_bin = int(angolo_phi // 5)
            conteggi_raggruppati[indice_bin][i] = conteggi_raggruppati[indice_bin][i] + conteggio
            conteggi_per_bin[indice_bin][i] += 1

# Calcola la media dei conteggi in ciascun bin
conteggi_raggruppati = conteggi_raggruppati[:conteggio_totale]
conteggi_per_bin = conteggi_per_bin[:conteggio_totale]

# Visualizza l'istogramma
plt.figure(figsize=(10, 6))
#plt.bar(bin_angolari, conteggi_raggruppati[:, 0]/conteggi_per_bin[:, 0], width=5, align='edge', alpha=.5, label="Bottom")
#plt.bar(bin_angolari, conteggi_raggruppati[:, 1]/conteggi_per_bin[:, 1], width=5, align='edge', alpha=.5, label="X1")
#plt.bar(bin_angolari, conteggi_raggruppati[:, 2]/conteggi_per_bin[:, 2], width=5, align='edge', alpha=.5, label="BGO_X1")
#plt.bar(bin_angolari, conteggi_raggruppati[:, 3]/conteggi_per_bin[:, 3], width=5, align='edge', alpha=.5, label="BGO_X0")
plt.bar(bin_angolari, conteggi_raggruppati[:, 4]/conteggi_per_bin[:, 4], width=5, align='edge', alpha=.5, label="BGO_Y1")
plt.bar(bin_angolari, conteggi_raggruppati[:, 5]/conteggi_per_bin[:, 5], width=5, align='edge', alpha=.5, label="BGO_Y0")

plt.title('')
plt.xlabel('Phi (°)', fontsize=15)
plt.ylabel('Conteggi medi per pannello', fontsize=15)
plt.tick_params(axis='both', labelsize=15) 
plt.legend(fontsize=15)
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# Define 5-degree bins
bins = np.arange(0, 185, 5)

# Group counts by 5-degree bins
grouped_counts = np.zeros((len(bins), number_of_detectors))
counts_per_bin = np.zeros((len(bins), number_of_detectors))

total_count = 0
for i in range(0, number_of_detectors):
    for theta, phi, count in zip(labels[:, 0], labels[:, 1], counts[:, i]):
        
        if (phi > 80 and phi < 110):
            total_count += 1
            bin_index = int(theta // 5)
            grouped_counts[bin_index][i] = grouped_counts[bin_index][i] + count
            counts_per_bin[bin_index][i] += 1

# Compute the mean counts in each bin
grouped_counts = grouped_counts[:total_count]
counts_per_bin = counts_per_bin[:total_count]

# Plot histogram
plt.figure(figsize=(10, 6))
plt.bar(bins, grouped_counts[:, 0] / counts_per_bin[:, 0], width=5, align='edge', alpha=.5, label="bottom 1")
plt.bar(bins, grouped_counts[:, 1] / counts_per_bin[:, 1], width=5, align='edge', alpha=.5, label="bottom 2")
#plt.bar(bins, grouped_counts[:, 2] / counts_per_bin[:, 2], width=5, align='edge', alpha=.5, label="x1")
#plt.bar(bins, grouped_counts[:, 3] / counts_per_bin[:, 3], width=5, align='edge', alpha=.5, label="x2")
#plt.bar(bins, grouped_counts[:, 4] / counts_per_bin[:, 4], width=5, align='edge', alpha=.5, label="y1")
#plt.bar(bins, grouped_counts[:, 5] / counts_per_bin[:, 5], width=5, align='edge', alpha=.5, label="y2")

plt.xlabel('Theta (°)')
plt.ylabel('Panel mean counts')
plt.title('Mean counts for panels phi=[80°, 110°]')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# Define 5-degree bins
bins = np.arange(0, 185, 5)

# Group counts by 5-degree bins
grouped_counts = np.zeros((len(bins), number_of_detectors))
counts_per_bin = np.zeros((len(bins), number_of_detectors))

total_count = 0
for i in range(0, number_of_detectors):
    for theta, phi, count in zip(labels[:, 0], labels[:, 1], counts[:, i]):
        
        if (phi > 35 and phi < 55):
            total_count += 1
            bin_index = int(theta // 5)
            grouped_counts[bin_index][i] = grouped_counts[bin_index][i] + count
            counts_per_bin[bin_index][i] += 1

# Compute the mean counts in each bin
grouped_counts = grouped_counts[:total_count]
counts_per_bin = counts_per_bin[:total_count]

# Plot histogram
plt.figure(figsize=(10, 6))
#plt.bar(bins, grouped_counts[:, 0] / counts_per_bin[:, 0], width=5, align='edge', alpha=.5, label="bottom 1")
#plt.bar(bins, grouped_counts[:, 1] / counts_per_bin[:, 1], width=5, align='edge', alpha=.5, label="bottom 2")
plt.bar(bins, grouped_counts[:, 2] / counts_per_bin[:, 2], width=5, align='edge', alpha=.5, label="x1")
plt.bar(bins, grouped_counts[:, 3] / counts_per_bin[:, 3], width=5, align='edge', alpha=.5, label="x2")
plt.bar(bins, grouped_counts[:, 4] / counts_per_bin[:, 4], width=5, align='edge', alpha=.5, label="y1")
plt.bar(bins, grouped_counts[:, 5] / counts_per_bin[:, 5], width=5, align='edge', alpha=.5, label="y2")

plt.xlabel('Theta (°)')
plt.ylabel('Panel mean counts')
plt.title('Mean counts for panels phi=[35°,55°]')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# Define 5-degree bins
bins = np.arange(0, 185, 5)

# Group counts by 5-degree bins
grouped_counts = np.zeros((len(bins), number_of_detectors))
counts_per_bin = np.zeros((len(bins), number_of_detectors))

total_count = 0
for i in range(0, number_of_detectors):
    for theta, phi, count in zip(labels[:, 0], labels[:, 1], counts[:, i]):
        
        if (phi > 35 and phi < 55):
            total_count += 1
            bin_index = int(theta // 5)
            grouped_counts[bin_index][i] = grouped_counts[bin_index][i] + count
            counts_per_bin[bin_index][i] += 1

# Compute the mean counts in each bin
grouped_counts = grouped_counts[:total_count]
counts_per_bin = counts_per_bin[:total_count]

# Plot histogram
plt.figure(figsize=(10, 6))
plt.bar(bins, grouped_counts[:, 0] / counts_per_bin[:, 0], width=5, align='edge', alpha=.5, label="bottom 1")
plt.bar(bins, grouped_counts[:, 1] / counts_per_bin[:, 1], width=5, align='edge', alpha=.5, label="bottom 2")
#plt.bar(bins, grouped_counts[:, 2] / counts_per_bin[:, 2], width=5, align='edge', alpha=.5, label="x1")
#plt.bar(bins, grouped_counts[:, 3] / counts_per_bin[:, 3], width=5, align='edge', alpha=.5, label="x2")
#plt.bar(bins, grouped_counts[:, 4] / counts_per_bin[:, 4], width=5, align='edge', alpha=.5, label="y1")
#plt.bar(bins, grouped_counts[:, 5] / counts_per_bin[:, 5], width=5, align='edge', alpha=.5, label="y2")

plt.xlabel('Theta (°)')
plt.ylabel('Panel mean counts')
plt.title('Mean counts for panels phi=[35°,55°]')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# Define 5-degree bins
bins = np.arange(0, 185, 5)

# Group counts by 5-degree bins
grouped_counts = np.zeros((len(bins), number_of_detectors))
counts_per_bin = np.zeros((len(bins), number_of_detectors))

total_count = 0
for i in range(0, number_of_detectors):
    for theta, phi, count in zip(labels[:, 0], labels[:, 1], counts[:, i]):
        
        if (phi > 170 and phi < 180):
            total_count += 1
            bin_index = int(theta // 5)
            grouped_counts[bin_index][i] = grouped_counts[bin_index][i] + count
            counts_per_bin[bin_index][i] += 1

# Compute the mean counts in each bin
grouped_counts = grouped_counts[:total_count]
counts_per_bin = counts_per_bin[:total_count]

# Plot histogram
plt.figure(figsize=(10, 6))
#plt.bar(bins, grouped_counts[:, 0] / counts_per_bin[:, 0], width=5, align='edge', alpha=.5, label="bottom 1")
#plt.bar(bins, grouped_counts[:, 1] / counts_per_bin[:, 1], width=5, align='edge', alpha=.5, label="bottom 2")
plt.bar(bins, grouped_counts[:, 2] / counts_per_bin[:, 2], width=5, align='edge', alpha=.5, label="x1")
plt.bar(bins, grouped_counts[:, 3] / counts_per_bin[:, 3], width=5, align='edge', alpha=.5, label="x2")
plt.bar(bins, grouped_counts[:, 4] / counts_per_bin[:, 4], width=5, align='edge', alpha=.5, label="y1")
plt.bar(bins, grouped_counts[:, 5] / counts_per_bin[:, 5], width=5, align='edge', alpha=.5, label="y2")

plt.xlabel('Theta (°)')
plt.ylabel('Panel mean counts')
plt.title('Mean counts for panels phi=[170°,180°]')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# Define 5-degree bins
bins = np.arange(0, 185, 5)

# Group counts by 5-degree bins
grouped_counts = np.zeros((len(bins), number_of_detectors))
counts_per_bin = np.zeros((len(bins), number_of_detectors))

total_count = 0
for i in range(0, number_of_detectors):
    for theta, phi, count in zip(labels[:, 0], labels[:, 1], counts[:, i]):
        
        if (phi > 170 and phi < 180):
            total_count += 1
            bin_index = int(theta // 5)
            grouped_counts[bin_index][i] = grouped_counts[bin_index][i] + count
            counts_per_bin[bin_index][i] += 1

# Compute the mean counts in each bin
grouped_counts = grouped_counts[:total_count]
counts_per_bin = counts_per_bin[:total_count]

# Plot histogram
plt.figure(figsize=(10, 6))
plt.bar(bins, grouped_counts[:, 0] / counts_per_bin[:, 0], width=5, align='edge', alpha=.5, label="bottom 1")
plt.bar(bins, grouped_counts[:, 1] / counts_per_bin[:, 1], width=5, align='edge', alpha=.5, label="bottom 2")
#plt.bar(bins, grouped_counts[:, 2] / counts_per_bin[:, 2], width=5, align='edge', alpha=.5, label="x1")
#plt.bar(bins, grouped_counts[:, 3] / counts_per_bin[:, 3], width=5, align='edge', alpha=.5, label="x2")
#plt.bar(bins, grouped_counts[:, 4] / counts_per_bin[:, 4], width=5, align='edge', alpha=.5, label="y1")
#plt.bar(bins, grouped_counts[:, 5] / counts_per_bin[:, 5], width=5, align='edge', alpha=.5, label="y2")

plt.xlabel('Theta (°)')
plt.ylabel('Panel mean counts')
plt.title('Mean counts for panels phi=[170°,180°]')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# Define 5-degree bins
bins = np.arange(0, 365, 5)

# Group counts by 5-degree bins
grouped_counts = np.zeros((len(bins), 6))
counts_per_bin = np.zeros((len(bins), 6))

total_count = 0
for i in range(0, 6):
    for theta, phi, count in zip(labels[:, 0], labels[:, 1], counts[:, i]):
        
        if (theta > 85 and theta < 95):
            total_count += 1
            bin_index = int(phi // 5)
            grouped_counts[bin_index][i] = grouped_counts[bin_index][i] + count
            counts_per_bin[bin_index][i] += 1

# Compute the mean counts in each bin
grouped_counts = grouped_counts[:total_count]
counts_per_bin = counts_per_bin[:total_count]

# Plot histogram
plt.figure(figsize=(10, 6))
#plt.bar(bins, grouped_counts[:, 0] / counts_per_bin[:, 0], width=5, align='edge', alpha=.5, label="bottom")
#plt.bar(bins, grouped_counts[:, 1] / counts_per_bin[:, 1], width=5, align='edge', alpha=.5, label="x1")
plt.bar(bins, grouped_counts[:, 2] / counts_per_bin[:, 2], width=5, align='edge', alpha=.5, label="BGO_X1")
plt.bar(bins, grouped_counts[:, 3] / counts_per_bin[:, 3], width=5, align='edge', alpha=.5, label="BGO_X0")
#plt.bar(bins, grouped_counts[:, 4] / counts_per_bin[:, 4], width=5, align='edge', alpha=.5, label="y")
#plt.bar(bins, grouped_counts[:, 5] / counts_per_bin[:, 5], width=5, align='edge', alpha=.5, label="y_negative")

plt.xlabel('Phi (°)')
plt.ylabel('Panel mean counts')
plt.title('Mean counts of each panel')
plt.grid(True)
plt.legend()
plt.show()
